In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

torvik_df = pd.read_csv("s3://collegebasketballinsiders/torvik/2026/team-ratings.csv")
map_df = pd.read_csv("s3://collegebasketballinsiders/general/map.csv", index_col=0)

In [2]:
torvik_df = torvik_df[['season', 'date', 'team', 'rank', 'conf', 'games',
       'adj_off_eff', 'adj_def_eff', 'barthag', 'efg_pct', 'efgd_pct', 'tor',
       'tord', 'orb', 'drb', 'ftr', 'ftrd', 'two_pt_pct', 'two_pt_def_pct',
       'three_pt_pct', 'three_pt_def_pct', 'three_pt_rt', 'three_pt_def_rt',
       'adj_tempo', 'wab']]
torvik_df["date"] = pd.to_datetime(torvik_df["date"], format="%Y%m%d")

torvik_df = torvik_df.merge(map_df[["team_id", "torrvik"]], left_on="team", right_on="torrvik").drop("torrvik", axis=1)

In [3]:
kenpom_df = pd.read_csv("s3://collegebasketballinsiders/kenpom/2026/team-ratings.csv")
kenpom_df = kenpom_df[['ArchiveDate', 'Season', 'TeamID', 'TeamName',
       'ConfShort', 'AdjEM', 'RankAdjEM', 'AdjOE', 'RankAdjOE',
       'AdjDE', 'RankAdjDE', 'AdjTempo', 'RankAdjTempo']]
kenpom_df.columns = ['date', 'season', 'team_id', 'team',
       'conf', 'adj_em', 'adj_em_rank', 'adj_oe', 'adj_oe_rank',
       'adj_def', 'adj_def_rank', 'adj_tempo', 'adj_tempo_rank']
kenpom_df["date"] = pd.to_datetime(kenpom_df["date"])
height_df = pd.read_csv("s3://collegebasketballinsiders/kenpom/2026/team-height.csv")
height_df = height_df.merge(map_df[["team_id", "kenpom"]], left_on="TeamName", right_on="kenpom")
height_df = height_df[['Season', 'team_id', 'TeamName', 'ConfShort', 
       'AvgHgt', 'AvgHgtRank', 'HgtEff', 'HgtEffRank', 'Hgt5', 'Hgt5Rank',
       'Hgt4', 'Hgt4Rank', 'Hgt3', 'Hgt3Rank', 'Hgt2', 'Hgt2Rank', 'Hgt1',
       'Hgt1Rank', 'Exp', 'ExpRank', 'Bench', 'BenchRank', 'Continuity',
       'RankContinuity']]
height_df.columns = ['season', 'team_id', 'team', 'conf',
       'avg_height', 'avg_height_rank', 'eff_height', 'eff_height_rank', 'height_5', 'height_5_rank',
       'height_4', 'height_4_rank', 'height_3', 'height_3_rank', 'height_2', 'height_2_rank', 'height_1',
       'height_1_rank', 'exp', 'exp_rank', 'bench', 'bench_rank', 'continuity',
       'continuity_rank']

In [4]:
rating_df = kenpom_df.merge(torvik_df, on=['date', 'team_id'], how="left")
rating_df['season'] = rating_df['season_x']
rating_df['conf'] = rating_df['conf_x']
rating_df['team'] = rating_df['team_x']
rating_df['adj_tempo_km'] = rating_df['adj_tempo_x']
rating_df['adj_tempo_tvk'] = rating_df['adj_tempo_y']
rating_df = rating_df[['date', 'season', 'team_id', 'team', 'conf', 'adj_em',
       'adj_em_rank', 'adj_oe', 'adj_oe_rank', 'adj_def', 'adj_def_rank',
       'adj_tempo_km', 'adj_tempo_rank','rank', 
       'games', 'adj_off_eff', 'adj_def_eff', 'barthag', 'efg_pct', 'efgd_pct',
       'tor', 'tord', 'orb', 'drb', 'ftr', 'ftrd', 'two_pt_pct',
       'two_pt_def_pct', 'three_pt_pct', 'three_pt_def_pct', 'three_pt_rt',
       'three_pt_def_rt', 'adj_tempo_tvk', 'wab']]

In [5]:
rating_df = rating_df.merge(height_df, on=['season', 'team_id'], how="left")
rating_df['team'] = rating_df['team_x']
rating_df['conf'] = rating_df['conf_x']
rating_df = rating_df[['date', 'season', 'team_id', 'team', 'conf', 'adj_em',
       'adj_em_rank', 'adj_oe', 'adj_oe_rank', 'adj_def', 'adj_def_rank',
       'adj_tempo_km', 'adj_tempo_rank', 'rank', 'games', 'adj_off_eff',
       'adj_def_eff', 'barthag', 'efg_pct', 'efgd_pct', 'tor', 'tord', 'orb',
       'drb', 'ftr', 'ftrd', 'two_pt_pct', 'two_pt_def_pct', 'three_pt_pct',
       'three_pt_def_pct', 'three_pt_rt', 'three_pt_def_rt', 'adj_tempo_tvk',
       'wab', 'avg_height', 'avg_height_rank',
       'eff_height', 'eff_height_rank', 'height_5', 'height_5_rank',
       'height_4', 'height_4_rank', 'height_3', 'height_3_rank', 'height_2',
       'height_2_rank', 'height_1', 'height_1_rank', 'exp', 'exp_rank',
       'bench', 'bench_rank', 'continuity', 'continuity_rank']]

In [6]:
def shift_ratings_one_game(rating_df: pd.DataFrame, make_new_cols=False) -> pd.DataFrame:
    """
    Shift per-game rating/stat columns back by 1 within (season, team_id),
    while leaving height/exp/bench/continuity + ID/meta columns unshifted.

    If make_new_cols=True, creates *_lag1 columns instead of overwriting.
    """
    df = rating_df.copy()

    # ---- ID/meta (never shift)
    id_cols = ["date", "season", "team_id", "team", "conf"]

    # ---- Non-shift stat families (explicit names from your schema)
    no_shift_cols = id_cols + [
        "avg_height", "avg_height_rank",
        "eff_height", "eff_height_rank",
        "height_5", "height_5_rank",
        "height_4", "height_4_rank",
        "height_3", "height_3_rank",
        "height_2", "height_2_rank",
        "height_1", "height_1_rank",
        "exp", "exp_rank",
        "bench", "bench_rank",
        "continuity", "continuity_rank",
    ]

    # ---- Columns to shift = everything else that exists in the df
    shift_cols = [c for c in df.columns if c not in set(no_shift_cols)]

    # Ensure date is datetime and sort for proper game order
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["season", "team_id", "date"])

    # Grouped lag
    lagged = df.groupby(["season", "team_id"], dropna=False)[shift_cols].shift(1)

    if make_new_cols:
        df[[f"{c}_lag1" for c in shift_cols]] = lagged
    else:
        df[shift_cols] = lagged

    return df

kp_tvk_features_df = shift_ratings_one_game(rating_df=rating_df)

In [7]:
kp_tvk_features_df = kp_tvk_features_df[~kp_tvk_features_df['adj_em'].isna()]

In [8]:
### ESPN GAME STATS
game_info_df = pd.read_csv("s3://collegebasketballinsiders/box-scores/2026/game-info/game-info.csv")
team_stats_df = pd.read_csv("s3://collegebasketballinsiders/box-scores/2026/teams/team-stats.csv")
todays_games_df = game_info_df[game_info_df['date_utc'] >= "2026-01-14"]
todays_games_df[['home_team', 'away_team', 'home_1h', 'away_1h', 'home_2h', 'away_2h', 'home_score',
       'away_score']] = todays_games_df[['away_team', 'home_team', 'away_1h', 'home_1h',  'away_2h', 'home_2h',
       'away_score','home_score']]

todays_games_df = todays_games_df.merge(map_df[["espn_2", "team_id"]], left_on="home_team", right_on="espn_2")
todays_games_df['home_team_id'] = todays_games_df['team_id']
todays_games_df = todays_games_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
todays_games_df = todays_games_df.merge(map_df[["espn_2", "team_id"]], left_on="away_team", right_on="espn_2")
todays_games_df['away_team_id'] = todays_games_df['team_id']
todays_games_df = todays_games_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
todays_games_df['season'] = 2026
todays_games_df = todays_games_df[['game_id', 'season', 'date_utc', 'time_utc', 'neutral_site', 'home_team',
       'home_team_id', 'away_team', 'away_team_id']]

In [9]:
game_info_df[['home_team', 'away_team', 'home_1h', 'away_1h', 'home_2h', 'away_2h', 'home_score',
       'away_score']] = game_info_df[['away_team', 'home_team', 'away_1h', 'home_1h',  'away_2h', 'home_2h',
       'away_score','home_score']]
game_info_df = game_info_df.merge(map_df[["espn_2", "team_id"]], left_on="home_team", right_on="espn_2")
game_info_df['home_team_id'] = game_info_df['team_id']
game_info_df = game_info_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
game_info_df = game_info_df.merge(map_df[["espn_2", "team_id"]], left_on="away_team", right_on="espn_2")
game_info_df['away_team_id'] = game_info_df['team_id']
game_info_df = game_info_df[['game_id', 'date_utc', 'time_utc', 'neutral_site',
       'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_1h', 'away_1h', 'home_2h', 'away_2h',
       'home_score', 'away_score']]
game_info_df = game_info_df[~game_info_df['home_2h'].isna()]

In [10]:
team_stats_df = team_stats_df.merge(map_df[["espn_2", "team_id"]], left_on="team", right_on="espn_2")
team_stats_df['team_id'] = team_stats_df['team_id_y']
team_stats_df = team_stats_df[['game_id', 'team', 'team_id', 'assists', 'defensiveRebounds', 'fouls',
       'totalRebounds', 
       'pointsInPaint', 'technicalFouls',
       'offensiveRebounds',  'turnoverPoints', 'steals', 'blocks', 'fastBreakPoints',
       'turnovers']]

In [11]:
def season_grab(df):
    """
    Assigns a season year based on date_utc.
    Example: games before 2021-04-01 belong to season 2021,
             games between 2021-04-01 and 2022-04-01 belong to 2022, etc.
    """
    # Ensure date_utc is datetime
    df = df.copy()
    df['date_utc'] = pd.to_datetime(df['date_utc'])

    # Define season cutoffs
    bins = [
        pd.Timestamp("1900-01-01"),
        pd.Timestamp("2021-04-21"),
        pd.Timestamp("2022-04-21"),
        pd.Timestamp("2023-04-21"),
        pd.Timestamp("2024-04-21"),
        pd.Timestamp("2025-04-21"),
        pd.Timestamp("2100-01-21"),
    ]
    seasons = [2021, 2022, 2023, 2024, 2025, 2026]

    # Use pandas cut to categorize efficiently
    df['season'] = pd.cut(df['date_utc'], bins=bins, labels=seasons, right=False).astype(int)

    return df
game_info_df = season_grab(game_info_df)


In [12]:
import pandas as pd
import numpy as np

# ======================
# Your existing config
# ======================
BASE_STATS = [
    'assists', 'defensiveRebounds', 'fouls', 'totalRebounds',
    'pointsInPaint', 'technicalFouls', 'offensiveRebounds',
    'turnoverPoints', 'steals', 'blocks', 'fastBreakPoints', 'turnovers'
]
ROLL_WINDOWS = [1, 3, 5, 10]

# --- Robust datetime combiner (handles "HHZ", "HH:MMZ", "HH:MM:SSZ" or without Z):
def _combine_utc_datetime(date_series: pd.Series, time_series: pd.Series) -> pd.Series:
    d = date_series.astype(str).str.strip()
    t = time_series.astype(str).str.strip().str.upper()
    # default blank -> noon UTC
    t = t.mask((t.eq("")) | t.isna(), "12:00:00Z")
    # strip Z, normalize to HH:MM:SS, then add Z back
    s = pd.Series(t.str.replace("Z", "", regex=False))
    s = s.where(~s.str.match(r"^\d{1,2}$"), s + ":00:00")   # HH -> HH:00:00
    s = s.where(~s.str.match(r"^\d{1,2}:\d{2}$"), s + ":00")# HH:MM -> HH:MM:00
    t_full = s.astype(str) + "Z"
    return pd.to_datetime(d + " " + t_full, errors="coerce", utc=True)

# ======================
# Your existing builders
# ======================

def build_team_stats_features_no_dup(
    game_info_df: pd.DataFrame,
    team_stats_df: pd.DataFrame,
    windows: list[int] = ROLL_WINDOWS,
    local_time_zone: str | None = None,
    agg: str | dict = "sum",
) -> pd.DataFrame:
    req_game = ['game_id','date_utc','time_utc','season',
                'home_team','home_team_id','away_team','away_team_id']
    missing = [c for c in req_game if c not in game_info_df.columns]
    if missing:
        raise KeyError(f"Missing in game_info_df: {missing}")

    if not {'game_id','team_id'}.issubset(team_stats_df.columns):
        raise KeyError("team_stats_df must include 'game_id' and 'team_id'.")

    stats = [s for s in BASE_STATS if s in team_stats_df.columns]
    if not stats:
        out = game_info_df.copy()
        out["game_datetime_utc"] = _combine_utc_datetime(out["date_utc"], out["time_utc"])
        out["game_datetime_local"] = (
            out["game_datetime_utc"].dt.tz_convert(local_time_zone)
            if local_time_zone else out["game_datetime_utc"]
        )
        return out

    agg_map = {s: agg if isinstance(agg, str) else agg.get(s, "sum") for s in stats}
    ts_clean = (
        team_stats_df
        .groupby(["game_id","team_id"], as_index=False)
        .agg(agg_map)
    )
    assert not ts_clean.duplicated(["game_id","team_id"]).any(), "Aggregation failed to ensure uniqueness."

    gif = game_info_df.copy()
    gif["game_datetime_utc"] = _combine_utc_datetime(gif["date_utc"], gif["time_utc"])
    gif["game_datetime_local"] = (
        gif["game_datetime_utc"].dt.tz_convert(local_time_zone)
        if local_time_zone else gif["game_datetime_utc"]
    )

    home_stats = ts_clean.rename(columns={"team_id":"home_team_id", **{s: f"{s}_home" for s in stats}})
    away_stats = ts_clean.rename(columns={"team_id":"away_team_id", **{s: f"{s}_away" for s in stats}})
    merged = (
        gif.merge(home_stats, on=["game_id","home_team_id"], how="left")
           .merge(away_stats, on=["game_id","away_team_id"], how="left")
    )

    home_payload = {
        'game_id': merged['game_id'],
        'season': merged['season'],
        'game_dt': merged['game_datetime_utc'],
        'team_id': merged['home_team_id'],
        'side': 'home'
    }
    for s in stats:
        home_payload[s] = merged.get(f"{s}_home")
        home_payload[f"allowed_{s}"] = merged.get(f"{s}_away")

    away_payload = {
        'game_id': merged['game_id'],
        'season': merged['season'],
        'game_dt': merged['game_datetime_utc'],
        'team_id': merged['away_team_id'],
        'side': 'away'
    }
    for s in stats:
        away_payload[s] = merged.get(f"{s}_away")
        away_payload[f"allowed_{s}"] = merged.get(f"{s}_home")

    long_team = pd.concat([pd.DataFrame(home_payload), pd.DataFrame(away_payload)], ignore_index=True)
    long_team = long_team.sort_values(['team_id','season','game_dt'], kind='mergesort')
    g = long_team.groupby(['team_id','season'], group_keys=False)

    for s in stats:
        for col in (s, f"allowed_{s}"):
            shifted = g[col].shift(1)  # use only past games
            for w in windows:
                long_team[f"ra{w}_{col}"] = (
                    long_team.assign(_s=shifted)
                             .groupby(['team_id','season'], group_keys=False)['_s']
                             .rolling(window=w, min_periods=1)
                             .mean()
                             .reset_index(level=[0,1], drop=True)
                )

    roll_cols = [c for c in long_team.columns if c.startswith("ra")]
    home_feats = (
        long_team[long_team['side']=='home'][['game_id','team_id'] + roll_cols]
        .rename(columns={'team_id':'home_team_id', **{c: f"home_{c}" for c in roll_cols}})
    )
    away_feats = (
        long_team[long_team['side']=='away'][['game_id','team_id'] + roll_cols]
        .rename(columns={'team_id':'away_team_id', **{c: f"away_{c}" for c in roll_cols}})
    )

    out = (
        merged.merge(home_feats, on=['game_id','home_team_id'], how='left')
              .merge(away_feats, on=['game_id','away_team_id'], how='left')
    )

    id_cols = ['game_id','date_utc','time_utc','season','home_team','home_team_id','away_team','away_team_id']
    roll_outs = [c for c in out.columns if c.startswith('home_ra') or c.startswith('away_ra')]
    ordered = [c for c in id_cols if c in out.columns] + roll_outs
    ordered += [c for c in out.columns if c not in ordered]
    return out[ordered]

# --- Game-level rolling (points/totals/margins/time features) you shared:
REQUIRED_COLS = [
    "game_id", "date_utc", "time_utc", "neutral_site",
    "home_team", "home_team_id", "away_team", "away_team_id",
    "home_1h", "away_1h", "home_2h", "away_2h",
    "home_score", "away_score", "season",
    "total", "1h_total", "2h_total",
    "margin", "1h_margin", "2h_margin",
]

def _to_flag(x) -> int:
    if pd.isna(x):
        return 0
    if isinstance(x, (int, float)) and not pd.isna(x):
        return int(x != 0)
    s = str(x).strip().lower()
    return int(s in {"1","true","t","y","yes","neutral","neutral_site"})

def build_cbb_features_multiroll(
    games_df: pd.DataFrame,
    windows: list[int] = [1, 3, 5, 10],
    local_time_zone: str | None = None,
) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLS if c not in games_df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df = games_df.copy()
    df["game_datetime_utc"] = _combine_utc_datetime(df["date_utc"], df["time_utc"])
    df["game_datetime_local"] = (
        df["game_datetime_utc"].dt.tz_convert(local_time_zone)
        if local_time_zone else df["game_datetime_utc"]
    )

    gdl = "game_datetime_local"
    df["game_dow"]        = df[gdl].dt.weekday
    df["game_month"]      = df[gdl].dt.month
    df["game_dayofyear"]  = df[gdl].dt.dayofyear
    df["game_weekofyear"] = df[gdl].dt.isocalendar().week.astype(int)
    df["game_hour"]       = df[gdl].dt.hour
    df["is_weekend"]      = df["game_dow"].isin([5, 6]).astype(int)
    df["neutral_site_flag"] = df["neutral_site"].apply(_to_flag).astype(int)

    if "1h_total" not in df or df["1h_total"].isna().all():
        df["1h_total"] = df["home_1h"] + df["away_1h"]
    if "2h_total" not in df or df["2h_total"].isna().all():
        df["2h_total"] = df["home_2h"] + df["away_2h"]
    if "total" not in df or df["total"].isna().all():
        df["total"] = df["home_score"] + df["away_score"]
    if "1h_margin" not in df or df["1h_margin"].isna().all():
        df["1h_margin"] = df["home_1h"] - df["away_1h"]
    if "2h_margin" not in df or df["2h_margin"].isna().all():
        df["2h_margin"] = df["home_2h"] - df["away_2h"]
    if "margin" not in df or df["margin"].isna().all():
        df["margin"] = df["home_score"] - df["away_score"]

    home_side = pd.DataFrame({
        "game_id":     df["game_id"],
        "season":      df["season"],
        "game_dt":     df["game_datetime_utc"],
        "team":        df["home_team"],
        "team_id":     df["home_team_id"],
        "opponent":    df["away_team"],
        "opponent_id": df["away_team_id"],
        "side":        "home",
        "pts_1h":      df["home_1h"],
        "pts_2h":      df["home_2h"],
        "pts_g":       df["home_score"],
        "opp_1h":      df["away_1h"],
        "opp_2h":      df["away_2h"],
        "opp_g":       df["away_score"],
        "is_neutral":  df["neutral_site_flag"].astype(int),
    })
    away_side = pd.DataFrame({
        "game_id":     df["game_id"],
        "season":      df["season"],
        "game_dt":     df["game_datetime_utc"],
        "team":        df["away_team"],
        "team_id":     df["away_team_id"],
        "opponent":    df["home_team"],
        "opponent_id": df["home_team_id"],
        "side":        "away",
        "pts_1h":      df["away_1h"],
        "pts_2h":      df["away_2h"],
        "pts_g":       df["away_score"],
        "opp_1h":      df["home_1h"],
        "opp_2h":      df["home_2h"],
        "opp_g":       df["home_score"],
        "is_neutral":  df["neutral_site_flag"].astype(int),
    })
    long_team = pd.concat([home_side, away_side], ignore_index=True)

    long_team["mgn_1h"] = long_team["pts_1h"] - long_team["opp_1h"]
    long_team["mgn_2h"] = long_team["pts_2h"] - long_team["opp_2h"]
    long_team["mgn_g"]  = long_team["pts_g"]  - long_team["opp_g"]
    long_team["tot_1h"] = long_team["pts_1h"] + long_team["opp_1h"]
    long_team["tot_g"]  = long_team["pts_g"]  + long_team["opp_g"]

    long_team = long_team.sort_values(["team_id", "season", "game_dt"], kind="mergesort")

    # days since last (this will compute rest for today's rows too)
    long_team["days_since_last_game"] = (
        long_team.groupby(["team_id", "season"], group_keys=False)["game_dt"]
                 .diff()
                 .dt.total_seconds()
                 .div(86400.0)
    )

    base_feats = {
        "pts_1h": "ra_pts_1h",
        "pts_2h": "ra_pts_2h",
        "pts_g":  "ra_pts_g",
        "opp_1h": "ra_allowed_1h",
        "opp_2h": "ra_allowed_2h",
        "opp_g":  "ra_allowed_g",
        "mgn_1h": "ra_mgn_1h",
        "mgn_2h": "ra_mgn_2h",
        "mgn_g":  "ra_mgn_g",
        "tot_1h": "ra_tot_1h",
        "tot_g":  "ra_tot_g",
    }
    g = long_team.groupby(["team_id", "season"], group_keys=False)
    for src_col, base_name in base_feats.items():
        shifted = g[src_col].shift(1)  # <-- leakage-safe
        long_team[f"__shifted__{src_col}"] = shifted
        for w in windows:
            long_team[f"{base_name}_{w}"] = (
                long_team.groupby(["team_id", "season"], group_keys=False)[f"__shifted__{src_col}"]
                        .rolling(window=w, min_periods=1)
                        .mean()
                        .reset_index(level=[0,1], drop=True)
            )
    long_team.drop(columns=[c for c in long_team.columns if c.startswith("__shifted__")], inplace=True)

    roll_cols = [c for c in long_team.columns if c.startswith("ra_")]
    keep_cols = ["game_id", "team", "team_id", "side", "days_since_last_game", "is_neutral"] + roll_cols

    home_feats = (
        long_team[long_team["side"] == "home"][keep_cols]
        .drop(columns=["side"])
        .rename(columns={
            "team": "home_team",
            "team_id": "home_team_id",
            "days_since_last_game": "home_days_since_last",
            "is_neutral": "home_is_neutral",
            **{c: f"home_{c}" for c in roll_cols}
        })
    )
    away_feats = (
        long_team[long_team["side"] == "away"][keep_cols]
        .drop(columns=["side"])
        .rename(columns={
            "team": "away_team",
            "team_id": "away_team_id",
            "days_since_last_game": "away_days_since_last",
            "is_neutral": "away_is_neutral",
            **{c: f"away_{c}" for c in roll_cols}
        })
    )

    out = (
        df.merge(home_feats, on=["game_id", "home_team", "home_team_id"], how="left")
          .merge(away_feats, on=["game_id", "away_team", "away_team_id"], how="left")
    )

    id_cols   = [
        "game_id","date_utc","time_utc","game_datetime_utc","game_datetime_local",
        "season","neutral_site","neutral_site_flag",
        "home_team","home_team_id","away_team","away_team_id"
    ]
    time_cols = ["game_dow","game_month","game_dayofyear","game_weekofyear","game_hour","is_weekend"]
    spacing   = ["home_days_since_last","away_days_since_last","home_is_neutral","away_is_neutral"]
    roll_outs = [c for c in out.columns if c.startswith("home_ra_") or c.startswith("away_ra_")]
    ordered = [c for c in id_cols + time_cols + spacing + roll_outs if c in out.columns]
    ordered += [c for c in out.columns if c not in ordered]
    return out[ordered]

# ============================================
# Minimal-tweak inference via simple append
# ============================================
def build_inference_features_via_append(
    todays_games_df: pd.DataFrame,
    hist_games_df: pd.DataFrame,
    team_stats_df: pd.DataFrame,
    local_time_zone: str | None = "America/New_York",
) -> pd.DataFrame:
    """
    Append today's games (with targets/boxscore stats set to NaN) to history,
    run the SAME builders (which already use shift(1)), then return rows for today.
    No asof merges needed.
    """
    # 1) Ensure today's has the columns training code expects
    today = todays_games_df.copy()
    today = today[['game_id','season','date_utc','time_utc','neutral_site',
                   'home_team','home_team_id','away_team','away_team_id']]

    # add target columns (NaN) so build_cbb_features_multiroll can run without error
    add_cols = ["home_1h","away_1h","home_2h","away_2h","home_score","away_score",
                "total","1h_total","2h_total","margin","1h_margin","2h_margin"]
    for c in add_cols:
        if c not in today.columns:
            today[c] = np.nan

    # 2) Align history to required cols (keep originals in hist)
    hist = hist_games_df.copy()
    # make sure all required columns exist in hist (if any of total/margins missing, builder recomputes anyway)
    for c in ["total","1h_total","2h_total","margin","1h_margin","2h_margin"]:
        if c not in hist.columns:
            hist[c] = np.nan

    # hist['game_id'] = hist['game_id'].astype('int')
    # today['game_id'] = today['game_id'].astype('int')
    # 3) Concatenate history + today (today last so time order is natural)
    combined = pd.concat([hist, today], ignore_index=True, sort=False)

    # 4) Build game-level rolling/time features on the combined frame
    game_feats_all = build_cbb_features_multiroll(combined, windows=ROLL_WINDOWS, local_time_zone=local_time_zone)

    # 5) Build team boxscore rolling features on the combined schedule
    # (team_stats_df only contains historical rows; today's games simply won't match -> NaN FOR/AGAINST, which is fine)
    ts_feats_all = build_team_stats_features_no_dup(
        game_info_df=combined[['game_id','date_utc','time_utc','season','home_team','home_team_id','away_team','away_team_id']],
        team_stats_df=team_stats_df,
        windows=ROLL_WINDOWS,
        local_time_zone=local_time_zone
    )

    # 6) Extract today's rows (by game_id) and merge both feature sets
    today_ids = set(today['game_id'].tolist())
    gf_today = game_feats_all[game_feats_all['game_id'].isin(today_ids)].copy()
    ts_today = ts_feats_all[ts_feats_all['game_id'].isin(today_ids)].copy()

    # Keep only the rolling cols from team-stats frame (avoid duplicate id/time cols)
    ts_roll_cols = [c for c in ts_today.columns if c.startswith('home_ra') or c.startswith('away_ra')]
    out = gf_today.merge(
        ts_today[['game_id','home_team_id','away_team_id'] + ts_roll_cols],
        on=['game_id','home_team_id','away_team_id'],
        how='left'
    )

    # 7) Return one row per today's game with all rolling/time features
    out = out.sort_values('game_datetime_utc').reset_index(drop=True)
    return out


In [13]:
game_features_df = build_inference_features_via_append(todays_games_df, game_info_df, team_stats_df).drop_duplicates(subset=["home_team", "away_team"])
game_features_df = game_features_df[game_features_df['game_id'].isin(list(todays_games_df['game_id']))]

In [14]:
game_features = game_features_df[[
       ## GAME INFO
       'game_id', 'date_utc', 'season', 'neutral_site', 'home_team',
       'home_team_id', 'away_team', 'away_team_id', 
       ## TARGETS
       'home_1h', 'away_1h','home_2h', 'away_2h', 'home_score', 'away_score', 'total',
       '1h_total', '2h_total', 'margin', '1h_margin', '2h_margin',
       ## GAME INFO FEATURES
       'neutral_site_flag', 'game_dow', 'game_month', 'game_dayofyear',
       'game_weekofyear', 'game_hour', 'is_weekend',
       ## TEAM GAME INFO FEATURES
       'home_days_since_last', 'away_days_since_last', 'home_ra_pts_1h_1',
       'home_ra_pts_1h_3', 'home_ra_pts_1h_5', 'home_ra_pts_1h_10',
       'home_ra_pts_2h_1', 'home_ra_pts_2h_3', 'home_ra_pts_2h_5',
       'home_ra_pts_2h_10', 'home_ra_pts_g_1', 'home_ra_pts_g_3',
       'home_ra_pts_g_5', 'home_ra_pts_g_10', 'home_ra_allowed_1h_1',
       'home_ra_allowed_1h_3', 'home_ra_allowed_1h_5',
       'home_ra_allowed_1h_10', 'home_ra_allowed_2h_1',
       'home_ra_allowed_2h_3', 'home_ra_allowed_2h_5',
       'home_ra_allowed_2h_10', 'home_ra_allowed_g_1',
       'home_ra_allowed_g_3', 'home_ra_allowed_g_5',
       'home_ra_allowed_g_10', 'home_ra_mgn_1h_1', 'home_ra_mgn_1h_3',
       'home_ra_mgn_1h_5', 'home_ra_mgn_1h_10', 'home_ra_mgn_2h_1',
       'home_ra_mgn_2h_3', 'home_ra_mgn_2h_5', 'home_ra_mgn_2h_10',
       'home_ra_mgn_g_1', 'home_ra_mgn_g_3', 'home_ra_mgn_g_5',
       'home_ra_mgn_g_10', 'home_ra_tot_1h_1', 'home_ra_tot_1h_3',
       'home_ra_tot_1h_5', 'home_ra_tot_1h_10', 'home_ra_tot_g_1',
       'home_ra_tot_g_3', 'home_ra_tot_g_5', 'home_ra_tot_g_10',
       'away_ra_pts_1h_1', 'away_ra_pts_1h_3', 'away_ra_pts_1h_5',
       'away_ra_pts_1h_10', 'away_ra_pts_2h_1', 'away_ra_pts_2h_3',
       'away_ra_pts_2h_5', 'away_ra_pts_2h_10', 'away_ra_pts_g_1',
       'away_ra_pts_g_3', 'away_ra_pts_g_5', 'away_ra_pts_g_10',
       'away_ra_allowed_1h_1', 'away_ra_allowed_1h_3',
       'away_ra_allowed_1h_5', 'away_ra_allowed_1h_10',
       'away_ra_allowed_2h_1', 'away_ra_allowed_2h_3',
       'away_ra_allowed_2h_5', 'away_ra_allowed_2h_10',
       'away_ra_allowed_g_1', 'away_ra_allowed_g_3',
       'away_ra_allowed_g_5', 'away_ra_allowed_g_10', 'away_ra_mgn_1h_1',
       'away_ra_mgn_1h_3', 'away_ra_mgn_1h_5', 'away_ra_mgn_1h_10',
       'away_ra_mgn_2h_1', 'away_ra_mgn_2h_3', 'away_ra_mgn_2h_5',
       'away_ra_mgn_2h_10', 'away_ra_mgn_g_1', 'away_ra_mgn_g_3',
       'away_ra_mgn_g_5', 'away_ra_mgn_g_10', 'away_ra_tot_1h_1',
       'away_ra_tot_1h_3', 'away_ra_tot_1h_5', 'away_ra_tot_1h_10',
       'away_ra_tot_g_1', 'away_ra_tot_g_3', 'away_ra_tot_g_5',
       'away_ra_tot_g_10', 
       ## TEAM GAME STATS FEATURES
       'home_ra1_assists', 'home_ra3_assists',
       'home_ra5_assists', 'home_ra10_assists',
       'home_ra1_allowed_assists', 'home_ra3_allowed_assists',
       'home_ra5_allowed_assists', 'home_ra10_allowed_assists',
       'home_ra1_defensiveRebounds', 'home_ra3_defensiveRebounds',
       'home_ra5_defensiveRebounds', 'home_ra10_defensiveRebounds',
       'home_ra1_allowed_defensiveRebounds',
       'home_ra3_allowed_defensiveRebounds',
       'home_ra5_allowed_defensiveRebounds',
       'home_ra10_allowed_defensiveRebounds', 'home_ra1_fouls',
       'home_ra3_fouls', 'home_ra5_fouls', 'home_ra10_fouls',
       'home_ra1_allowed_fouls', 'home_ra3_allowed_fouls',
       'home_ra5_allowed_fouls', 'home_ra10_allowed_fouls',
       'home_ra1_totalRebounds', 'home_ra3_totalRebounds',
       'home_ra5_totalRebounds', 'home_ra10_totalRebounds',
       'home_ra1_allowed_totalRebounds', 'home_ra3_allowed_totalRebounds',
       'home_ra5_allowed_totalRebounds',
       'home_ra10_allowed_totalRebounds', 'home_ra1_pointsInPaint',
       'home_ra3_pointsInPaint', 'home_ra5_pointsInPaint',
       'home_ra10_pointsInPaint', 'home_ra1_allowed_pointsInPaint',
       'home_ra3_allowed_pointsInPaint', 'home_ra5_allowed_pointsInPaint',
       'home_ra10_allowed_pointsInPaint', 'home_ra1_technicalFouls',
       'home_ra3_technicalFouls', 'home_ra5_technicalFouls',
       'home_ra10_technicalFouls', 'home_ra1_allowed_technicalFouls',
       'home_ra3_allowed_technicalFouls',
       'home_ra5_allowed_technicalFouls',
       'home_ra10_allowed_technicalFouls', 'home_ra1_offensiveRebounds',
       'home_ra3_offensiveRebounds', 'home_ra5_offensiveRebounds',
       'home_ra10_offensiveRebounds',
       'home_ra1_allowed_offensiveRebounds',
       'home_ra3_allowed_offensiveRebounds',
       'home_ra5_allowed_offensiveRebounds',
       'home_ra10_allowed_offensiveRebounds', 'home_ra1_turnoverPoints',
       'home_ra3_turnoverPoints', 'home_ra5_turnoverPoints',
       'home_ra10_turnoverPoints', 'home_ra1_allowed_turnoverPoints',
       'home_ra3_allowed_turnoverPoints',
       'home_ra5_allowed_turnoverPoints',
       'home_ra10_allowed_turnoverPoints', 'home_ra1_steals',
       'home_ra3_steals', 'home_ra5_steals', 'home_ra10_steals',
       'home_ra1_allowed_steals', 'home_ra3_allowed_steals',
       'home_ra5_allowed_steals', 'home_ra10_allowed_steals',
       'home_ra1_blocks', 'home_ra3_blocks', 'home_ra5_blocks',
       'home_ra10_blocks', 'home_ra1_allowed_blocks',
       'home_ra3_allowed_blocks', 'home_ra5_allowed_blocks',
       'home_ra10_allowed_blocks', 'home_ra1_fastBreakPoints',
       'home_ra3_fastBreakPoints', 'home_ra5_fastBreakPoints',
       'home_ra10_fastBreakPoints', 'home_ra1_allowed_fastBreakPoints',
       'home_ra3_allowed_fastBreakPoints',
       'home_ra5_allowed_fastBreakPoints',
       'home_ra10_allowed_fastBreakPoints', 'home_ra1_turnovers',
       'home_ra3_turnovers', 'home_ra5_turnovers', 'home_ra10_turnovers',
       'home_ra1_allowed_turnovers', 'home_ra3_allowed_turnovers',
       'home_ra5_allowed_turnovers', 'home_ra10_allowed_turnovers',
       'away_ra1_assists', 'away_ra3_assists', 'away_ra5_assists',
       'away_ra10_assists', 'away_ra1_allowed_assists',
       'away_ra3_allowed_assists', 'away_ra5_allowed_assists',
       'away_ra10_allowed_assists', 'away_ra1_defensiveRebounds',
       'away_ra3_defensiveRebounds', 'away_ra5_defensiveRebounds',
       'away_ra10_defensiveRebounds',
       'away_ra1_allowed_defensiveRebounds',
       'away_ra3_allowed_defensiveRebounds',
       'away_ra5_allowed_defensiveRebounds',
       'away_ra10_allowed_defensiveRebounds', 'away_ra1_fouls',
       'away_ra3_fouls', 'away_ra5_fouls', 'away_ra10_fouls',
       'away_ra1_allowed_fouls', 'away_ra3_allowed_fouls',
       'away_ra5_allowed_fouls', 'away_ra10_allowed_fouls',
       'away_ra1_totalRebounds', 'away_ra3_totalRebounds',
       'away_ra5_totalRebounds', 'away_ra10_totalRebounds',
       'away_ra1_allowed_totalRebounds', 'away_ra3_allowed_totalRebounds',
       'away_ra5_allowed_totalRebounds',
       'away_ra10_allowed_totalRebounds', 'away_ra1_pointsInPaint',
       'away_ra3_pointsInPaint', 'away_ra5_pointsInPaint',
       'away_ra10_pointsInPaint', 'away_ra1_allowed_pointsInPaint',
       'away_ra3_allowed_pointsInPaint', 'away_ra5_allowed_pointsInPaint',
       'away_ra10_allowed_pointsInPaint', 'away_ra1_technicalFouls',
       'away_ra3_technicalFouls', 'away_ra5_technicalFouls',
       'away_ra10_technicalFouls', 'away_ra1_allowed_technicalFouls',
       'away_ra3_allowed_technicalFouls',
       'away_ra5_allowed_technicalFouls',
       'away_ra10_allowed_technicalFouls', 'away_ra1_offensiveRebounds',
       'away_ra3_offensiveRebounds', 'away_ra5_offensiveRebounds',
       'away_ra10_offensiveRebounds',
       'away_ra1_allowed_offensiveRebounds',
       'away_ra3_allowed_offensiveRebounds',
       'away_ra5_allowed_offensiveRebounds',
       'away_ra10_allowed_offensiveRebounds', 'away_ra1_turnoverPoints',
       'away_ra3_turnoverPoints', 'away_ra5_turnoverPoints',
       'away_ra10_turnoverPoints', 'away_ra1_allowed_turnoverPoints',
       'away_ra3_allowed_turnoverPoints',
       'away_ra5_allowed_turnoverPoints',
       'away_ra10_allowed_turnoverPoints', 'away_ra1_steals',
       'away_ra3_steals', 'away_ra5_steals', 'away_ra10_steals',
       'away_ra1_allowed_steals', 'away_ra3_allowed_steals',
       'away_ra5_allowed_steals', 'away_ra10_allowed_steals',
       'away_ra1_blocks', 'away_ra3_blocks', 'away_ra5_blocks',
       'away_ra10_blocks', 'away_ra1_allowed_blocks',
       'away_ra3_allowed_blocks', 'away_ra5_allowed_blocks',
       'away_ra10_allowed_blocks', 'away_ra1_fastBreakPoints',
       'away_ra3_fastBreakPoints', 'away_ra5_fastBreakPoints',
       'away_ra10_fastBreakPoints', 'away_ra1_allowed_fastBreakPoints',
       'away_ra3_allowed_fastBreakPoints',
       'away_ra5_allowed_fastBreakPoints',
       'away_ra10_allowed_fastBreakPoints', 'away_ra1_turnovers',
       'away_ra3_turnovers', 'away_ra5_turnovers', 'away_ra10_turnovers',
       'away_ra1_allowed_turnovers', 'away_ra3_allowed_turnovers',
       'away_ra5_allowed_turnovers', 'away_ra10_allowed_turnovers']]


In [15]:
kp_tvk_features_df = kp_tvk_features_df.sort_values(["date","team_id"], ascending=True).drop_duplicates(subset="team_id", keep="last")

In [16]:
def merge_team_ratings(features_df: pd.DataFrame, team_ratings_df: pd.DataFrame) -> pd.DataFrame:
    # --- Merge home team ratings
    merged = features_df.merge(
        team_ratings_df.add_suffix("_home"),
        how="left",
        left_on="home_team_id",
        right_on="team_id_home"
    )

    # --- Merge away team ratings
    merged = merged.merge(
        team_ratings_df.add_suffix("_away"),
        how="left",
        left_on="away_team_id",
        right_on="team_id_away"
    )

    # --- Optional: drop duplicate key columns introduced by the merges
    drop_cols = [
        "team_id_home","team_id_away"
    ]
    drop_cols = [c for c in drop_cols if c in merged.columns]
    merged = merged.drop(columns=drop_cols)

    return merged

inference_df = merge_team_ratings(game_features, kp_tvk_features_df)

In [17]:
inference_df = inference_df[[
       ## GAME INFO
       'game_id', 'date_utc', 'season', 'neutral_site', 'home_team',
       'home_team_id', 'conf_home', 'away_team', 'away_team_id', 'conf_away',
       ## GAME INFO FEATURES
       'neutral_site_flag', 'game_dow', 'game_month', 'game_dayofyear',
       'game_weekofyear', 'game_hour', 'is_weekend',
       ## TEAM GAME INFO FEATURES
       'home_days_since_last', 'away_days_since_last', 'home_ra_pts_1h_1',
       'home_ra_pts_1h_3', 'home_ra_pts_1h_5', 'home_ra_pts_1h_10',
       'home_ra_pts_2h_1', 'home_ra_pts_2h_3', 'home_ra_pts_2h_5',
       'home_ra_pts_2h_10', 'home_ra_pts_g_1', 'home_ra_pts_g_3',
       'home_ra_pts_g_5', 'home_ra_pts_g_10', 'home_ra_allowed_1h_1',
       'home_ra_allowed_1h_3', 'home_ra_allowed_1h_5',
       'home_ra_allowed_1h_10', 'home_ra_allowed_2h_1',
       'home_ra_allowed_2h_3', 'home_ra_allowed_2h_5',
       'home_ra_allowed_2h_10', 'home_ra_allowed_g_1',
       'home_ra_allowed_g_3', 'home_ra_allowed_g_5',
       'home_ra_allowed_g_10', 'home_ra_mgn_1h_1', 'home_ra_mgn_1h_3',
       'home_ra_mgn_1h_5', 'home_ra_mgn_1h_10', 'home_ra_mgn_2h_1',
       'home_ra_mgn_2h_3', 'home_ra_mgn_2h_5', 'home_ra_mgn_2h_10',
       'home_ra_mgn_g_1', 'home_ra_mgn_g_3', 'home_ra_mgn_g_5',
       'home_ra_mgn_g_10', 'home_ra_tot_1h_1', 'home_ra_tot_1h_3',
       'home_ra_tot_1h_5', 'home_ra_tot_1h_10', 'home_ra_tot_g_1',
       'home_ra_tot_g_3', 'home_ra_tot_g_5', 'home_ra_tot_g_10',
       'away_ra_pts_1h_1', 'away_ra_pts_1h_3', 'away_ra_pts_1h_5',
       'away_ra_pts_1h_10', 'away_ra_pts_2h_1', 'away_ra_pts_2h_3',
       'away_ra_pts_2h_5', 'away_ra_pts_2h_10', 'away_ra_pts_g_1',
       'away_ra_pts_g_3', 'away_ra_pts_g_5', 'away_ra_pts_g_10',
       'away_ra_allowed_1h_1', 'away_ra_allowed_1h_3',
       'away_ra_allowed_1h_5', 'away_ra_allowed_1h_10',
       'away_ra_allowed_2h_1', 'away_ra_allowed_2h_3',
       'away_ra_allowed_2h_5', 'away_ra_allowed_2h_10',
       'away_ra_allowed_g_1', 'away_ra_allowed_g_3',
       'away_ra_allowed_g_5', 'away_ra_allowed_g_10', 'away_ra_mgn_1h_1',
       'away_ra_mgn_1h_3', 'away_ra_mgn_1h_5', 'away_ra_mgn_1h_10',
       'away_ra_mgn_2h_1', 'away_ra_mgn_2h_3', 'away_ra_mgn_2h_5',
       'away_ra_mgn_2h_10', 'away_ra_mgn_g_1', 'away_ra_mgn_g_3',
       'away_ra_mgn_g_5', 'away_ra_mgn_g_10', 'away_ra_tot_1h_1',
       'away_ra_tot_1h_3', 'away_ra_tot_1h_5', 'away_ra_tot_1h_10',
       'away_ra_tot_g_1', 'away_ra_tot_g_3', 'away_ra_tot_g_5',
       'away_ra_tot_g_10', 
       ## TEAM GAME STATS FEATURES
       'home_ra1_assists', 'home_ra3_assists',
       'home_ra5_assists', 'home_ra10_assists',
       'home_ra1_allowed_assists', 'home_ra3_allowed_assists',
       'home_ra5_allowed_assists', 'home_ra10_allowed_assists',
       'home_ra1_defensiveRebounds', 'home_ra3_defensiveRebounds',
       'home_ra5_defensiveRebounds', 'home_ra10_defensiveRebounds',
       'home_ra1_allowed_defensiveRebounds',
       'home_ra3_allowed_defensiveRebounds',
       'home_ra5_allowed_defensiveRebounds',
       'home_ra10_allowed_defensiveRebounds', 'home_ra1_fouls',
       'home_ra3_fouls', 'home_ra5_fouls', 'home_ra10_fouls',
       'home_ra1_allowed_fouls', 'home_ra3_allowed_fouls',
       'home_ra5_allowed_fouls', 'home_ra10_allowed_fouls',
       'home_ra1_totalRebounds', 'home_ra3_totalRebounds',
       'home_ra5_totalRebounds', 'home_ra10_totalRebounds',
       'home_ra1_allowed_totalRebounds', 'home_ra3_allowed_totalRebounds',
       'home_ra5_allowed_totalRebounds',
       'home_ra10_allowed_totalRebounds', 'home_ra1_pointsInPaint',
       'home_ra3_pointsInPaint', 'home_ra5_pointsInPaint',
       'home_ra10_pointsInPaint', 'home_ra1_allowed_pointsInPaint',
       'home_ra3_allowed_pointsInPaint', 'home_ra5_allowed_pointsInPaint',
       'home_ra10_allowed_pointsInPaint', 'home_ra1_technicalFouls',
       'home_ra3_technicalFouls', 'home_ra5_technicalFouls',
       'home_ra10_technicalFouls', 'home_ra1_allowed_technicalFouls',
       'home_ra3_allowed_technicalFouls',
       'home_ra5_allowed_technicalFouls',
       'home_ra10_allowed_technicalFouls', 'home_ra1_offensiveRebounds',
       'home_ra3_offensiveRebounds', 'home_ra5_offensiveRebounds',
       'home_ra10_offensiveRebounds',
       'home_ra1_allowed_offensiveRebounds',
       'home_ra3_allowed_offensiveRebounds',
       'home_ra5_allowed_offensiveRebounds',
       'home_ra10_allowed_offensiveRebounds', 'home_ra1_turnoverPoints',
       'home_ra3_turnoverPoints', 'home_ra5_turnoverPoints',
       'home_ra10_turnoverPoints', 'home_ra1_allowed_turnoverPoints',
       'home_ra3_allowed_turnoverPoints',
       'home_ra5_allowed_turnoverPoints',
       'home_ra10_allowed_turnoverPoints', 'home_ra1_steals',
       'home_ra3_steals', 'home_ra5_steals', 'home_ra10_steals',
       'home_ra1_allowed_steals', 'home_ra3_allowed_steals',
       'home_ra5_allowed_steals', 'home_ra10_allowed_steals',
       'home_ra1_blocks', 'home_ra3_blocks', 'home_ra5_blocks',
       'home_ra10_blocks', 'home_ra1_allowed_blocks',
       'home_ra3_allowed_blocks', 'home_ra5_allowed_blocks',
       'home_ra10_allowed_blocks', 'home_ra1_fastBreakPoints',
       'home_ra3_fastBreakPoints', 'home_ra5_fastBreakPoints',
       'home_ra10_fastBreakPoints', 'home_ra1_allowed_fastBreakPoints',
       'home_ra3_allowed_fastBreakPoints',
       'home_ra5_allowed_fastBreakPoints',
       'home_ra10_allowed_fastBreakPoints', 'home_ra1_turnovers',
       'home_ra3_turnovers', 'home_ra5_turnovers', 'home_ra10_turnovers',
       'home_ra1_allowed_turnovers', 'home_ra3_allowed_turnovers',
       'home_ra5_allowed_turnovers', 'home_ra10_allowed_turnovers',
       'away_ra1_assists', 'away_ra3_assists', 'away_ra5_assists',
       'away_ra10_assists', 'away_ra1_allowed_assists',
       'away_ra3_allowed_assists', 'away_ra5_allowed_assists',
       'away_ra10_allowed_assists', 'away_ra1_defensiveRebounds',
       'away_ra3_defensiveRebounds', 'away_ra5_defensiveRebounds',
       'away_ra10_defensiveRebounds',
       'away_ra1_allowed_defensiveRebounds',
       'away_ra3_allowed_defensiveRebounds',
       'away_ra5_allowed_defensiveRebounds',
       'away_ra10_allowed_defensiveRebounds', 'away_ra1_fouls',
       'away_ra3_fouls', 'away_ra5_fouls', 'away_ra10_fouls',
       'away_ra1_allowed_fouls', 'away_ra3_allowed_fouls',
       'away_ra5_allowed_fouls', 'away_ra10_allowed_fouls',
       'away_ra1_totalRebounds', 'away_ra3_totalRebounds',
       'away_ra5_totalRebounds', 'away_ra10_totalRebounds',
       'away_ra1_allowed_totalRebounds', 'away_ra3_allowed_totalRebounds',
       'away_ra5_allowed_totalRebounds',
       'away_ra10_allowed_totalRebounds', 'away_ra1_pointsInPaint',
       'away_ra3_pointsInPaint', 'away_ra5_pointsInPaint',
       'away_ra10_pointsInPaint', 'away_ra1_allowed_pointsInPaint',
       'away_ra3_allowed_pointsInPaint', 'away_ra5_allowed_pointsInPaint',
       'away_ra10_allowed_pointsInPaint', 'away_ra1_technicalFouls',
       'away_ra3_technicalFouls', 'away_ra5_technicalFouls',
       'away_ra10_technicalFouls', 'away_ra1_allowed_technicalFouls',
       'away_ra3_allowed_technicalFouls',
       'away_ra5_allowed_technicalFouls',
       'away_ra10_allowed_technicalFouls', 'away_ra1_offensiveRebounds',
       'away_ra3_offensiveRebounds', 'away_ra5_offensiveRebounds',
       'away_ra10_offensiveRebounds',
       'away_ra1_allowed_offensiveRebounds',
       'away_ra3_allowed_offensiveRebounds',
       'away_ra5_allowed_offensiveRebounds',
       'away_ra10_allowed_offensiveRebounds', 'away_ra1_turnoverPoints',
       'away_ra3_turnoverPoints', 'away_ra5_turnoverPoints',
       'away_ra10_turnoverPoints', 'away_ra1_allowed_turnoverPoints',
       'away_ra3_allowed_turnoverPoints',
       'away_ra5_allowed_turnoverPoints',
       'away_ra10_allowed_turnoverPoints', 'away_ra1_steals',
       'away_ra3_steals', 'away_ra5_steals', 'away_ra10_steals',
       'away_ra1_allowed_steals', 'away_ra3_allowed_steals',
       'away_ra5_allowed_steals', 'away_ra10_allowed_steals',
       'away_ra1_blocks', 'away_ra3_blocks', 'away_ra5_blocks',
       'away_ra10_blocks', 'away_ra1_allowed_blocks',
       'away_ra3_allowed_blocks', 'away_ra5_allowed_blocks',
       'away_ra10_allowed_blocks', 'away_ra1_fastBreakPoints',
       'away_ra3_fastBreakPoints', 'away_ra5_fastBreakPoints',
       'away_ra10_fastBreakPoints', 'away_ra1_allowed_fastBreakPoints',
       'away_ra3_allowed_fastBreakPoints',
       'away_ra5_allowed_fastBreakPoints',
       'away_ra10_allowed_fastBreakPoints', 'away_ra1_turnovers',
       'away_ra3_turnovers', 'away_ra5_turnovers', 'away_ra10_turnovers',
       'away_ra1_allowed_turnovers', 'away_ra3_allowed_turnovers',
       'away_ra5_allowed_turnovers', 'away_ra10_allowed_turnovers',
       ## RATING FEATURES
       'adj_em_home','adj_em_rank_home', 'adj_oe_home', 'adj_oe_rank_home',
       'adj_def_home', 'adj_def_rank_home', 'adj_tempo_km_home',
       'adj_tempo_rank_home', 'rank_home', 'games_home',
       'adj_off_eff_home', 'adj_def_eff_home', 'barthag_home',
       'efg_pct_home', 'efgd_pct_home', 'tor_home', 'tord_home',
       'orb_home', 'drb_home', 'ftr_home', 'ftrd_home', 'two_pt_pct_home',
       'two_pt_def_pct_home', 'three_pt_pct_home',
       'three_pt_def_pct_home', 'three_pt_rt_home',
       'three_pt_def_rt_home', 'adj_tempo_tvk_home', 'wab_home',
       'avg_height_home', 'avg_height_rank_home', 'eff_height_home',
       'eff_height_rank_home', 'height_5_home', 'height_5_rank_home',
       'height_4_home', 'height_4_rank_home', 'height_3_home',
       'height_3_rank_home', 'height_2_home', 'height_2_rank_home',
       'height_1_home', 'height_1_rank_home', 'exp_home', 'exp_rank_home',
       'bench_home', 'bench_rank_home', 'continuity_home',
       'continuity_rank_home','adj_em_away', 'adj_em_rank_away', 'adj_oe_away',
       'adj_oe_rank_away', 'adj_def_away', 'adj_def_rank_away',
       'adj_tempo_km_away', 'adj_tempo_rank_away', 'rank_away',
       'games_away', 'adj_off_eff_away', 'adj_def_eff_away',
       'barthag_away', 'efg_pct_away', 'efgd_pct_away', 'tor_away',
       'tord_away', 'orb_away', 'drb_away', 'ftr_away', 'ftrd_away',
       'two_pt_pct_away', 'two_pt_def_pct_away', 'three_pt_pct_away',
       'three_pt_def_pct_away', 'three_pt_rt_away',
       'three_pt_def_rt_away', 'adj_tempo_tvk_away', 'wab_away',
       'avg_height_away', 'avg_height_rank_away', 'eff_height_away',
       'eff_height_rank_away', 'height_5_away', 'height_5_rank_away',
       'height_4_away', 'height_4_rank_away', 'height_3_away',
       'height_3_rank_away', 'height_2_away', 'height_2_rank_away',
       'height_1_away', 'height_1_rank_away', 'exp_away', 'exp_rank_away',
       'bench_away', 'bench_rank_away', 'continuity_away',
       'continuity_rank_away']]

In [18]:
import os
import numpy as np
import pandas as pd
from joblib import load as joblib_load
from typing import List

# ======================================================
# Sklearn compatibility patch for legacy artifacts
# ======================================================

def _patch_sklearn_for_legacy_artifacts():
    """
    Patch sklearn internals so that models/preprocessors pickled under
    different sklearn versions can be safely loaded & used.

    - Adds _RemainderColsList to sklearn.compose._column_transformer if missing
    - Adds _drop_idx_after_grouping to OneHotEncoder if missing
    """
    # Patch ColumnTransformer remainder helper
    try:
        from sklearn.compose import _column_transformer as _ct
        if not hasattr(_ct, "_RemainderColsList"):
            class _RemainderColsList(list):
                pass
            _ct._RemainderColsList = _RemainderColsList
    except Exception:
        pass

    # Patch OneHotEncoder private attribute used in newer sklearn
    try:
        from sklearn.preprocessing import _encoders as _enc
        OHE = _enc.OneHotEncoder
        if not hasattr(OHE, "_drop_idx_after_grouping"):
            OHE._drop_idx_after_grouping = None
    except Exception:
        try:
            from sklearn.preprocessing import OneHotEncoder as PublicOHE
            if not hasattr(PublicOHE, "_drop_idx_after_grouping"):
                PublicOHE._drop_idx_after_grouping = None
        except Exception:
            pass

# run patch immediately
_patch_sklearn_for_legacy_artifacts()


# ======================================================
# Paths / config
# ======================================================

REG_OUT_DIR   = "../predicting/training_reports"            # regression models
CLASS_OUT_DIR = "../predicting/training_reports_winprob"    # win-probability models

# Regression targets
REG_TARGETS: List[str] = [
    "home_1h","away_1h","home_2h","away_2h",
    "home_score","away_score",
    "total","1h_total","2h_total",
    "margin","1h_margin","2h_margin",
]

# Classification targets (ALL 4 win-prob models)
CLASS_TARGETS: List[str] = [
    "home_win",
    "away_win",
    "home_1h_win",
    "away_1h_win",
]

# Final output columns
ID_VIEW = [
    "game_id","date_utc","home_team","home_team_id","away_team","away_team_id"
]

DATE_COL = "date_utc"


# ======================================================
# Helpers
# ======================================================

def _robust_joblib_load(path: str):
    _patch_sklearn_for_legacy_artifacts()
    try:
        return joblib_load(path)
    except AttributeError:
        _patch_sklearn_for_legacy_artifacts()
        return joblib_load(path)

def _load_artifact(target: str, base_dir: str):
    path = os.path.join(base_dir, target, f"{target}_final_model.joblib")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing model artifact for {target}: {path}")

    art = _robust_joblib_load(path)

    for k in ["preprocessor", "model", "features"]:
        if k not in art:
            raise ValueError(f"Artifact for {target} missing key: '{k}'")

    # calibrator is optional (for backward compatibility)
    # if not present, we'll fall back to raw predict_proba
    return art

def _ensure_columns(df: pd.DataFrame, expected_cols: List[str]) -> pd.DataFrame:
    X = df.copy()
    for c in expected_cols:
        if c not in X.columns:
            X[c] = np.nan
    return X[expected_cols]


# ======================================================
# Regression prediction block (unchanged)
# ======================================================

def _predict_regression_targets(inference_df, preds):
    for target in REG_TARGETS:
        print(f"[predict][reg] {target} …")

        art = _load_artifact(target, REG_OUT_DIR)
        pre = art["preprocessor"]
        model = art["model"]
        feat_cols = art["features"]

        X_raw = _ensure_columns(inference_df, feat_cols)
        X_t = pre.transform(X_raw)

        expected = getattr(model, "n_features_in_", None)
        if expected and X_t.shape[1] != expected:
            raise ValueError(
                f"[{target}] Feature mismatch: model expects {expected}, got {X_t.shape[1]}"
            )

        preds[f"pred_{target}"] = model.predict(X_t).astype(float)

    return preds


# ======================================================
# Classification (win-probability) prediction block
# with calibration
# ======================================================

def _predict_classification_targets(inference_df, preds):
    for target in CLASS_TARGETS:
        print(f"[predict][cls] {target} …")

        art = _load_artifact(target, CLASS_OUT_DIR)
        pre = art["preprocessor"]
        model = art["model"]
        feat_cols = art["features"]
        calibrator = art.get("calibrator", None)  # may be None for older models

        X_raw = _ensure_columns(inference_df, feat_cols)
        X_t = pre.transform(X_raw)

        expected = getattr(model, "n_features_in_", None)
        if expected and X_t.shape[1] != expected:
            raise ValueError(
                f"[{target}] Feature mismatch: model expects {expected}, got {X_t.shape[1]}"
            )

        # --- Get probabilities, using calibration if available
        if calibrator is not None:
            # Use raw scores from LightGBM, then calibrate
            raw = model.predict(X_t, raw_score=True).reshape(-1, 1)
            proba = calibrator.predict_proba(raw)[:, 1].astype(float)
        else:
            # Fallback for legacy artifacts without calibrator
            proba = model.predict_proba(X_t)[:, 1].astype(float)

        label = (proba >= 0.5).astype(int)

        preds[f"proba_{target}"] = proba
        preds[f"pred_{target}"] = label

    return preds


# ======================================================
# Public API
# ======================================================

def predict_with_artifacts(inference_df: pd.DataFrame) -> pd.DataFrame:
    """Runs BOTH regression + classification models and uses calibrated win probs if available."""

    missing = [c for c in ID_VIEW if c not in inference_df.columns]
    if missing:
        raise KeyError(f"inference_df missing ID columns: {missing}")

    preds = inference_df[ID_VIEW].copy()

    # 1. Regression outputs
    preds = _predict_regression_targets(inference_df, preds)

    # 2. Win-prob outputs (INCLUDING 1H win prob models; calibrated if possible)
    preds = _predict_classification_targets(inference_df, preds)

    try:
        preds = preds.sort_values("date_utc").reset_index(drop=True)
    except Exception:
        pass

    return preds


In [19]:
final_predictions = predict_with_artifacts(inference_df)


[predict][reg] home_1h …
[LightGBM] [Warning] lambda_l1 is set=2.442322575734447e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.442322575734447e-07
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.9045551296476547, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9045551296476547
[LightGBM] [Warning] bagging_fraction is set=0.7296890365747699, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7296890365747699
[LightGBM] [Warning] lambda_l2 is set=1.891030945934824e-07, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.891030945934824e-07
[LightGBM] [Warning] min_data_in_leaf is set=124, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=124
[predict][reg] away_1h …
[LightGBM] [Warning] lambda_l1 is set=2.442322575734447e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.442322575734447e-07

In [20]:
date = "20260114"
game_ids = list(pd.read_csv(f"s3://collegebasketballinsiders/daily-box-score-ids/{date}/game_ids.csv")['game_id'])

final_predictions = final_predictions[final_predictions['game_id'].isin(game_ids)]
final_predictions.to_csv(f"s3://collegebasketballinsiders/predictions/{date}/trans_preds.csv")

In [21]:
final_predictions[["home_team", "away_team", "pred_margin"]]

,home_team,away_team,pred_margin
30,East Tennessee State Buccaneers,Western Carolina Catamounts,2.931632
31,Chattanooga Mocs,Wofford Terriers,6.190540
32,Army Black Knights,Holy Cross Crusaders,4.227312
33,Lehigh Mountain Hawks,Boston University Terriers,6.989069
34,VCU Rams,Rhode Island Rams,4.136808
...,...,...,...
86,Nevada Wolf Pack,Utah State Aggies,6.531879
87,Arizona State Sun Devils,Arizona Wildcats,9.911618
88,Michigan Wolverines,Washington Huskies,6.711156
89,TCU Horned Frogs,BYU Cougars,6.471304


In [22]:
preds = pd.read_csv(f"s3://collegebasketballinsiders/predictions/{date}/preds.csv")

In [23]:
preds[["home_team", "away_team", "pred_margin"]]

,home_team,away_team,pred_margin
0,Rhode Island Rams,VCU Rams,5.825887
1,Western Carolina Catamounts,East Tennessee State Buccaneers,7.935387
2,Boston University Terriers,Lehigh Mountain Hawks,2.910905
3,Holy Cross Crusaders,Army Black Knights,5.022371
4,Wofford Terriers,Chattanooga Mocs,11.103675
...,...,...,...
56,Oregon State Beavers,Loyola Marymount Lions,4.625550
57,Arizona Wildcats,Arizona State Sun Devils,2.342746
58,Washington Huskies,Michigan Wolverines,9.289390
59,BYU Cougars,TCU Horned Frogs,8.367660
